In [ ]:
# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DOS GRÁFICOS
# ---------------------------------------------------------------------

from pathlib import Path
import textwrap

import matplotlib.pyplot as plt
import pandas as pd


# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent
ANALYTICS_DIR = SCRIPT_DIR.parent

OUTPUTS_DIR = ANALYTICS_DIR / "outputs" / "gold_03"
GRAFICOS_DIR = ANALYTICS_DIR / "graficos" / "gold_03"

GRAFICOS_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# ARQUIVOS DE ENTRADA
# ---------------------------------------------------------------------

ARQUIVO_COMPOSICAO_GENERO = OUTPUTS_DIR / "composicao_genero_historica.csv"
ARQUIVO_SENIORIDADE = OUTPUTS_DIR / "participacao_feminina_senioridade_historica.csv"
ARQUIVO_CARGO = OUTPUTS_DIR / "participacao_feminina_cargo_historica.csv"
ARQUIVO_SALARIO = OUTPUTS_DIR / "participacao_feminina_salario_historica.csv"
ARQUIVO_COR_RACA = OUTPUTS_DIR / "composicao_cor_raca_historica.csv"
ARQUIVO_FEMININO_COR_RACA = OUTPUTS_DIR / "participacao_feminina_cor_raca_historica.csv"

arquivos_necessarios = [
    ARQUIVO_COMPOSICAO_GENERO,
    ARQUIVO_SENIORIDADE,
    ARQUIVO_CARGO,
    ARQUIVO_SALARIO,
    ARQUIVO_COR_RACA,
    ARQUIVO_FEMININO_COR_RACA
]

for arquivo in arquivos_necessarios:
    if not arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")


# ---------------------------------------------------------------------
# CARREGAR OUTPUTS
# ---------------------------------------------------------------------

df_composicao_genero = pd.read_csv(ARQUIVO_COMPOSICAO_GENERO)
df_senioridade = pd.read_csv(ARQUIVO_SENIORIDADE)
df_cargo = pd.read_csv(ARQUIVO_CARGO)
df_salario = pd.read_csv(ARQUIVO_SALARIO)
df_cor_raca = pd.read_csv(ARQUIVO_COR_RACA)
df_feminino_cor_raca = pd.read_csv(ARQUIVO_FEMININO_COR_RACA)


# ---------------------------------------------------------------------
# ORDENS PADRÃO
# ---------------------------------------------------------------------

ORDEM_EDICOES = [
    "2023-2024",
    "2024-2025",
    "2025-2026"
]

ORDEM_GENEROS = [
    "Masculino",
    "Feminino",
    "Outro",
    "Prefiro não informar"
]

ORDEM_NIVEIS = [
    "Júnior",
    "Pleno",
    "Sênior",
    "Especialista/Staff"
]

ORDEM_FAIXAS = [
    "Menos de R$ 1.000/mês",
    "de R$ 1.001/mês a R$ 2.000/mês",
    "de R$ 2.001/mês a R$ 3.000/mês",
    "de R$ 3.001/mês a R$ 4.000/mês",
    "de R$ 4.001/mês a R$ 6.000/mês",
    "de R$ 6.001/mês a R$ 8.000/mês",
    "de R$ 8.001/mês a R$ 12.000/mês",
    "de R$ 12.001/mês a R$ 16.000/mês",
    "de R$ 16.001/mês a R$ 20.000/mês",
    "de R$ 20.001/mês a R$ 25.000/mês",
    "de R$ 25.001/mês a R$ 30.000/mês",
    "de R$ 30.001/mês a R$ 40.000/mês",
    "Acima de R$ 40.001/mês"
]

ORDEM_COR_RACA = [
    "Branca",
    "Parda",
    "Preta",
    "Amarela"
]


# ---------------------------------------------------------------------
# FUNÇÕES VISUAIS
# ---------------------------------------------------------------------

def finalizar_grafico(fig, ax, nome_arquivo, grade=True):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if grade:
        ax.grid(axis="y", alpha=0.18)
        ax.set_axisbelow(True)

    plt.savefig(
        GRAFICOS_DIR / nome_arquivo,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


def quebrar_texto(texto, largura=32):
    return "\n".join(textwrap.wrap(str(texto), largura))


# ---------------------------------------------------------------------
# COMPOSIÇÃO GERAL DE GÊNERO
# ---------------------------------------------------------------------

pivot_genero = (
    df_composicao_genero
    .pivot(index="edicao", columns="genero", values="pct_geral")
    .reindex(ORDEM_EDICOES)
    .reindex(columns=ORDEM_GENEROS)
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(13, 7))

pivot_genero.plot(
    kind="bar",
    stacked=True,
    width=0.62,
    ax=ax
)

for container in ax.containers:
    labels = [
        f"{valor:.1f}%" if valor >= 5 else ""
        for valor in container.datavalues
    ]

    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=11
    )

fig.suptitle(
    "Composição de gênero dos respondentes",
    x=0.07,
    y=0.98,
    ha="left",
    fontsize=18,
    fontweight="bold"
)

fig.text(
    0.07,
    0.92,
    "A participação feminina permaneceu próxima de 24% nas duas primeiras edições e recuou para 22,7% em 2025-2026.",
    ha="left",
    fontsize=11
)

ax.set_xlabel("")
ax.set_ylabel("Participação (%)", fontsize=11)

ax.set_xticklabels(
    ORDEM_EDICOES,
    rotation=0,
    fontsize=11
)

ax.legend(
    title="Gênero",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout(rect=[0.05, 0.05, 0.84, 0.86])

finalizar_grafico(
    fig,
    ax,
    "01_composicao_genero.png"
)


# ---------------------------------------------------------------------
# PARTICIPAÇÃO FEMININA POR SENIORIDADE
# ---------------------------------------------------------------------

df_senioridade_plot = (
    df_senioridade
    .dropna(subset=["nivel", "pct_feminino"])
    .copy()
)

POSICAO_NIVEL = {
    nivel: posicao
    for posicao, nivel in enumerate(ORDEM_NIVEIS)
}

df_senioridade_plot["posicao_nivel"] = (
    df_senioridade_plot["nivel"].map(POSICAO_NIVEL)
)

df_senioridade_plot = (
    df_senioridade_plot
    .dropna(subset=["posicao_nivel"])
)

offset_senioridade = {
    "2023-2024": (-8, 10),
    "2024-2025": (0, -16),
    "2025-2026": (8, 10)
}

fig, ax = plt.subplots(figsize=(13, 7))

for edicao in ORDEM_EDICOES:
    dados = (
        df_senioridade_plot[
            df_senioridade_plot["edicao"] == edicao
        ]
        .sort_values("posicao_nivel")
    )

    ax.plot(
        dados["posicao_nivel"],
        dados["pct_feminino"],
        marker="o",
        linewidth=2.5,
        markersize=8,
        label=edicao
    )

    dx, dy = offset_senioridade[edicao]

    for _, linha in dados.iterrows():
        ax.annotate(
            f"{linha['pct_feminino']:.1f}%",
            (
                linha["posicao_nivel"],
                linha["pct_feminino"]
            ),
            xytext=(dx, dy),
            textcoords="offset points",
            ha="center",
            fontsize=9
        )

fig.suptitle(
    "Participação feminina por senioridade",
    x=0.07,
    y=0.98,
    ha="left",
    fontsize=18,
    fontweight="bold"
)

fig.text(
    0.07,
    0.92,
    "Na edição mais recente, a participação feminina cai de 28,2% no nível Júnior para 20,1% entre Especialistas/Staff.",
    ha="left",
    fontsize=11
)

ax.set_xticks(range(len(ORDEM_NIVEIS)))

ax.set_xticklabels(
    ORDEM_NIVEIS,
    fontsize=11
)

ax.set_xlabel("")
ax.set_ylabel("Participação feminina (%)", fontsize=11)

ax.legend(
    title="Edição",
    bbox_to_anchor=(1.01, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout(rect=[0.05, 0.05, 0.86, 0.86])

finalizar_grafico(
    fig,
    ax,
    "02_participacao_feminina_senioridade.png"
)


# ---------------------------------------------------------------------
# PARTICIPAÇÃO FEMININA POR CARGO | 2025-2026
# ---------------------------------------------------------------------

df_cargo_atual = (
    df_cargo[
        df_cargo["edicao"] == "2025-2026"
    ]
    .dropna(subset=["cargo_harmonizado", "pct_feminino"])
    .sort_values("pct_feminino", ascending=True)
)

fig, ax = plt.subplots(figsize=(15, 9))

barras = ax.barh(
    range(len(df_cargo_atual)),
    df_cargo_atual["pct_feminino"]
)

ax.set_yticks(range(len(df_cargo_atual)))

ax.set_yticklabels(
    [
        quebrar_texto(cargo, 36)
        for cargo in df_cargo_atual["cargo_harmonizado"]
    ],
    fontsize=10
)

ax.bar_label(
    barras,
    labels=[
        f"{valor:.1f}%"
        for valor in df_cargo_atual["pct_feminino"]
    ],
    padding=5,
    fontsize=10
)

ax.set_xlim(
    0,
    df_cargo_atual["pct_feminino"].max() + 8
)

fig.suptitle(
    "Participação feminina por cargo | 2025-2026",
    x=0.04,
    y=0.98,
    ha="left",
    fontsize=18,
    fontweight="bold"
)

fig.text(
    0.04,
    0.935,
    "A presença feminina varia entre as carreiras, com menor participação em funções de Engenharia, Machine Learning e Suporte.",
    ha="left",
    fontsize=11
)

ax.set_xlabel("Participação feminina (%)", fontsize=11)
ax.set_ylabel("")

plt.tight_layout(rect=[0.03, 0.05, 0.98, 0.90])

finalizar_grafico(
    fig,
    ax,
    "03_participacao_feminina_cargo_atual.png",
    grade=False
)


# ---------------------------------------------------------------------
# VARIAÇÃO DA PARTICIPAÇÃO FEMININA POR CARGO
# ---------------------------------------------------------------------

pivot_cargo = (
    df_cargo
    .pivot(
        index="cargo_harmonizado",
        columns="edicao",
        values="pct_feminino"
    )
)

pivot_cargo["variacao_pp"] = (
    pivot_cargo["2025-2026"]
    - pivot_cargo["2023-2024"]
)

variacao_cargo = (
    pivot_cargo
    .dropna(subset=["2023-2024", "2025-2026"])
    .sort_values("variacao_pp", ascending=True)
)

fig, ax = plt.subplots(figsize=(15, 9))

barras = ax.barh(
    range(len(variacao_cargo)),
    variacao_cargo["variacao_pp"]
)

ax.set_yticks(range(len(variacao_cargo)))

ax.set_yticklabels(
    [
        quebrar_texto(cargo, 35)
        for cargo in variacao_cargo.index
    ],
    fontsize=10
)

ax.axvline(0, linewidth=1)

amplitude = max(
    abs(variacao_cargo["variacao_pp"].min()),
    abs(variacao_cargo["variacao_pp"].max())
)

margem = max(
    amplitude * 0.04,
    0.4
)

for barra, valor in zip(
    barras,
    variacao_cargo["variacao_pp"]
):
    y = barra.get_y() + barra.get_height() / 2

    if valor >= 0:
        ax.text(
            valor + margem,
            y,
            f"{valor:+.1f} p.p.",
            va="center",
            ha="left",
            fontsize=9
        )

    else:
        ax.text(
            valor - margem,
            y,
            f"{valor:+.1f} p.p.",
            va="center",
            ha="right",
            fontsize=9
        )

ax.set_xlim(
    variacao_cargo["variacao_pp"].min() - 3,
    variacao_cargo["variacao_pp"].max() + 3
)

fig.suptitle(
    "Variação da participação feminina por cargo",
    x=0.04,
    y=0.98,
    ha="left",
    fontsize=18,
    fontweight="bold"
)

fig.text(
    0.04,
    0.935,
    "Entre 2023-2024 e 2025-2026, a evolução foi desigual: alguns cargos avançaram enquanto outros apresentaram retração.",
    ha="left",
    fontsize=11
)

ax.set_xlabel("Variação da participação feminina (p.p.)", fontsize=11)
ax.set_ylabel("")

plt.tight_layout(rect=[0.03, 0.05, 0.98, 0.90])

finalizar_grafico(
    fig,
    ax,
    "04_variacao_participacao_feminina_cargo.png",
    grade=False
)


# ---------------------------------------------------------------------
# PARTICIPAÇÃO FEMININA POR FAIXA SALARIAL
# ---------------------------------------------------------------------

faixas_nao_reconhecidas = sorted(
    set(
        df_salario["faixa_salarial"]
        .dropna()
    )
    - set(ORDEM_FAIXAS)
)

if faixas_nao_reconhecidas:
    print("\nFAIXAS SALARIAIS NÃO PLOTADAS:")

    for faixa in faixas_nao_reconhecidas:
        print(f"- {faixa}")

df_salario_plot = (
    df_salario[
        df_salario["faixa_salarial"].isin(ORDEM_FAIXAS)
    ]
    .dropna(subset=["faixa_salarial", "pct_feminino"])
    .copy()
)

POSICAO_FAIXA = {
    faixa: posicao
    for posicao, faixa in enumerate(ORDEM_FAIXAS)
}

df_salario_plot["posicao_faixa"] = (
    df_salario_plot["faixa_salarial"].map(POSICAO_FAIXA)
)

offset_salario = {
    "2023-2024": (-7, 11),
    "2024-2025": (0, -17),
    "2025-2026": (7, 11)
}

fig, ax = plt.subplots(figsize=(18, 9))

for edicao in ORDEM_EDICOES:
    dados = (
        df_salario_plot[
            df_salario_plot["edicao"] == edicao
        ]
        .sort_values("posicao_faixa")
    )

    ax.plot(
        dados["posicao_faixa"],
        dados["pct_feminino"],
        marker="o",
        linewidth=2.3,
        markersize=7,
        label=edicao
    )

    dx, dy = offset_salario[edicao]

    for _, linha in dados.iterrows():
        ax.annotate(
            f"{linha['pct_feminino']:.1f}%",
            (
                linha["posicao_faixa"],
                linha["pct_feminino"]
            ),
            xytext=(dx, dy),
            textcoords="offset points",
            ha="center",
            va="center",
            fontsize=7.5,
            bbox=dict(
                boxstyle="round,pad=0.15",
                facecolor="white",
                edgecolor="none",
                alpha=0.75
            )
        )

fig.suptitle(
    "Participação feminina por faixa salarial",
    x=0.05,
    y=0.98,
    ha="left",
    fontsize=18,
    fontweight="bold"
)

fig.text(
    0.05,
    0.925,
    "Nas três edições, a participação feminina tende a ser menor nas faixas salariais mais elevadas.",
    ha="left",
    fontsize=11
)

ax.set_xticks(range(len(ORDEM_FAIXAS)))

ax.set_xticklabels(
    [
        quebrar_texto(faixa, 18)
        for faixa in ORDEM_FAIXAS
    ],
    rotation=40,
    ha="right",
    fontsize=9
)

ax.set_xlabel("")
ax.set_ylabel("Participação feminina (%)", fontsize=11)

ax.legend(
    title="Edição",
    bbox_to_anchor=(1.01, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout(rect=[0.04, 0.11, 0.88, 0.87])

finalizar_grafico(
    fig,
    ax,
    "05_participacao_feminina_faixa_salarial.png"
)


# ---------------------------------------------------------------------
# PRESENÇA DE COR/RAÇA/ETNIA
# ---------------------------------------------------------------------

df_cor_raca_plot = (
    df_cor_raca[
        df_cor_raca["cor_raca_etnia"].isin(ORDEM_COR_RACA)
    ]
    .copy()
)

pivot_cor_raca = (
    df_cor_raca_plot
    .pivot(
        index="cor_raca_etnia",
        columns="edicao",
        values="pct_respondentes"
    )
    .reindex(ORDEM_COR_RACA)
    .reindex(columns=ORDEM_EDICOES)
)

fig, ax = plt.subplots(figsize=(13, 7))

pivot_cor_raca.plot(
    kind="bar",
    ax=ax,
    width=0.72
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=4,
        fontsize=9
    )

fig.suptitle(
    "Presença por cor/raça/etnia",
    x=0.07,
    y=0.98,
    ha="left",
    fontsize=18,
    fontweight="bold"
)

fig.text(
    0.07,
    0.92,
    "A presença das principais categorias permaneceu relativamente estável ao longo das três edições.",
    ha="left",
    fontsize=11
)

fig.text(
    0.07,
    0.02,
    "Nota: a dimensão admite múltiplas marcações; os percentuais não precisam somar 100%.",
    ha="left",
    fontsize=9
)

ax.set_xlabel("")
ax.set_ylabel("Respondentes associados à categoria (%)", fontsize=11)

ax.set_xticklabels(
    ORDEM_COR_RACA,
    rotation=0,
    fontsize=11
)

ax.legend(
    title="Edição",
    bbox_to_anchor=(1.01, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout(rect=[0.05, 0.08, 0.86, 0.86])

finalizar_grafico(
    fig,
    ax,
    "06_composicao_cor_raca.png"
)


# ---------------------------------------------------------------------
# PARTICIPAÇÃO FEMININA POR COR/RAÇA/ETNIA
# ---------------------------------------------------------------------

df_feminino_cor_raca_plot = (
    df_feminino_cor_raca[
        df_feminino_cor_raca["cor_raca_etnia"].isin(ORDEM_COR_RACA)
    ]
    .copy()
)

pivot_feminino_cor_raca = (
    df_feminino_cor_raca_plot
    .pivot(
        index="cor_raca_etnia",
        columns="edicao",
        values="pct_feminino"
    )
    .reindex(ORDEM_COR_RACA)
    .reindex(columns=ORDEM_EDICOES)
)

fig, ax = plt.subplots(figsize=(13, 7))

pivot_feminino_cor_raca.plot(
    kind="bar",
    ax=ax,
    width=0.70
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=4,
        fontsize=9
    )

ax.set_ylim(
    0,
    pivot_feminino_cor_raca.max().max() + 7
)

fig.suptitle(
    "Participação feminina por cor/raça/etnia",
    x=0.07,
    y=0.98,
    ha="left",
    fontsize=18,
    fontweight="bold"
)

fig.text(
    0.07,
    0.92,
    "A participação feminina recuou em todas as categorias raciais comparáveis entre 2023-2024 e 2025-2026.",
    ha="left",
    fontsize=11
)

ax.set_xlabel("")
ax.set_ylabel("Participação feminina (%)", fontsize=11)

ax.set_xticklabels(
    ORDEM_COR_RACA,
    rotation=0,
    fontsize=11
)

ax.legend(
    title="Edição",
    bbox_to_anchor=(1.01, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout(rect=[0.05, 0.05, 0.86, 0.86])

finalizar_grafico(
    fig,
    ax,
    "07_participacao_feminina_cor_raca.png"
)